# ParsBERT Persian POS Training

**FarsiSyntax Lab**

Research notebook for fine-tuning ParsBERT for Universal POS tagging.

## Research context

The project compares neural POS tagging on formal and informal Persian.

**Important:** The original notebook was executed in its research environment. Some code cells reference Google Colab paths or external datasets. Adjust these paths and install dependencies before rerunning the experiment.

Execution outputs have been removed from this publication copy. Original training logs are retained locally. See the repository README and reports for recorded evaluation results.


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 02
# COLAB CELL 1: GPU VERIFICATION
# ============================================

import torch
import sys

print("=== GOOGLE COLAB ENVIRONMENT ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

    properties = torch.cuda.get_device_properties(0)

    print(
        "GPU memory:",
        round(properties.total_memory / 1024**3, 2),
        "GB"
    )

    device = torch.device("cuda")

    # Verify a GPU computation
    x = torch.randn(1000, 1000, device=device)
    result = x @ x

    print("GPU computation:", "SUCCESS")
    print("\nGPU RUNTIME READY")

else:
    print("\nGPU NOT AVAILABLE")
    print("Change runtime type to GPU and reconnect.")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 02
# COLAB CELL 2: UPLOAD TRAINING DATA
# ============================================

from google.colab import files
from pathlib import Path
import zipfile
import json
import io

# 1. Upload ZIP from Windows
print("Select parsbert_colab_bundle.zip")

uploaded = files.upload()

filename = "parsbert_colab_bundle.zip"

assert filename in uploaded, (
    "Please upload parsbert_colab_bundle.zip"
)

# 2. Define Colab project directory
DATA_DIR = Path(
    "/content/farsi-syntax-lab/data/processed"
)

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# 3. Validate archive and extract files
expected_files = {
    "formal_train.jsonl",
    "formal_dev.jsonl",
    "informal_dev.jsonl",
    "metadata.json"
}

with zipfile.ZipFile(
    io.BytesIO(uploaded[filename])
) as archive:

    assert set(archive.namelist()) == expected_files
    assert archive.testzip() is None

    metadata = json.loads(
        archive.read("metadata.json")
    )

    assert len(metadata["pos2id"]) == 16

    for name in sorted(expected_files - {"metadata.json"}):

        sentence_count = 0
        word_count = 0

        with archive.open(name) as source:
            for line in source:

                example = json.loads(line)

                assert (
                    len(example["input_ids"])
                    == len(example["attention_mask"])
                    == len(example["labels"])
                )

                assert len(example["input_ids"]) <= 256

                sentence_count += 1

                word_count += sum(
                    label != -100
                    for label in example["labels"]
                )

        expected = metadata["datasets"][name]

        assert sentence_count == expected["sentences"]
        assert word_count == expected["labeled_words"]

        print(
            f"[OK] {name}: "
            f"{sentence_count:,} sentences, "
            f"{word_count:,} labeled words"
        )

    archive.extractall(DATA_DIR)

# 4. Confirm readiness
print("\n=== COLAB DATASET VERIFICATION ===")
print("Location:", DATA_DIR)
print("Model:", metadata["model_name"])
print("POS categories:", len(metadata["pos2id"]))

print("\nCOLAB DATASET READY")
print("Training started: NO")
print("Test datasets: NOT INCLUDED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 02
# COLAB CELL 3: INITIALIZE PARSBERT
# ============================================

import sys
import subprocess

# --------------------------------------------
# 1. Install compatible NLP packages
# --------------------------------------------

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "accelerate>=1.10,<2",
    "scikit-learn",
    "safetensors"
])

# --------------------------------------------
# 2. Import libraries
# --------------------------------------------

import json
import torch
import transformers
import accelerate

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    set_seed
)

# --------------------------------------------
# 3. Read configuration from uploaded bundle
# --------------------------------------------

with open(
    DATA_DIR / "metadata.json",
    "r",
    encoding="utf-8"
) as f:
    metadata = json.load(f)

MODEL_NAME = metadata["model_name"]
pos2id = metadata["pos2id"]

id2pos = {
    index: tag
    for tag, index in pos2id.items()
}

NUM_TAGS = len(pos2id)

assert NUM_TAGS == 16
assert torch.cuda.is_available()

set_seed(42)

device = torch.device("cuda")

print("=== ENVIRONMENT ===")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------
# 4. Load pretrained ParsBERT tokenizer
# --------------------------------------------

print("\n=== LOADING PARSBERT TOKENIZER ===")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

assert tokenizer.is_fast

print("Tokenizer vocabulary:", tokenizer.vocab_size)

# --------------------------------------------
# 5. Initialize POS classification model
# --------------------------------------------

print("\n=== LOADING PRETRAINED PARSBERT ===")

bert_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_TAGS,
    id2label=id2pos,
    label2id=pos2id
)

bert_model = bert_model.to(device)

total_parameters = sum(
    p.numel() for p in bert_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in bert_model.parameters()
    if p.requires_grad
)

print("\n=== MODEL ARCHITECTURE ===")
print("Model:", MODEL_NAME)
print("POS categories:", NUM_TAGS)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Device:", next(bert_model.parameters()).device)

# --------------------------------------------
# 6. Verify GPU forward pass
# --------------------------------------------

with open(
    DATA_DIR / "formal_train.jsonl",
    "r",
    encoding="utf-8"
) as f:
    sample = json.loads(f.readline())

input_ids = torch.tensor(
    [sample["input_ids"]],
    dtype=torch.long,
    device=device
)

attention_mask = torch.tensor(
    [sample["attention_mask"]],
    dtype=torch.long,
    device=device
)

bert_model.eval()

with torch.inference_mode():
    outputs = bert_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

assert outputs.logits.shape == (
    1,
    input_ids.shape[1],
    NUM_TAGS
)

print("\n=== GPU FORWARD PASS ===")
print("Input shape:", tuple(input_ids.shape))
print("Output shape:", tuple(outputs.logits.shape))
print("GPU forward pass: SUCCESS")

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print("\nPARSBERT MODEL INITIALIZATION COMPLETE")
print("Training started: NO")
print("Test datasets: NOT USED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 02
# COLAB CELL 4: DATASETS + DRIVE STORAGE
# ============================================

import json
import torch

from pathlib import Path
from google.colab import drive
from torch.utils.data import Dataset
from transformers import DataCollatorForTokenClassification

# --------------------------------------------
# 1. Mount Google Drive
# --------------------------------------------

print("=== CONNECTING GOOGLE DRIVE ===")

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FarsiSyntaxLab"
)

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
RESULTS_DIR = PROJECT_DIR / "results"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)


# --------------------------------------------
# 2. Load pre-tokenized POS datasets
# --------------------------------------------

class PersianTokenDataset(Dataset):

    def __init__(self, path):

        self.examples = []

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                self.examples.append(json.loads(line))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


train_dataset = PersianTokenDataset(
    DATA_DIR / "formal_train.jsonl"
)

formal_eval_dataset = PersianTokenDataset(
    DATA_DIR / "formal_dev.jsonl"
)

informal_eval_dataset = PersianTokenDataset(
    DATA_DIR / "informal_dev.jsonl"
)


# --------------------------------------------
# 3. Verify dataset lengths and label counts
# --------------------------------------------

print("\n=== DATASET VERIFICATION ===")

expected = {
    "Formal train": (train_dataset, 26196, 452496),
    "Formal dev": (formal_eval_dataset, 1456, 25147),
    "Informal dev": (informal_eval_dataset, 598, 10049)
}

for name, (dataset, expected_sentences, expected_words) in expected.items():

    labeled_words = sum(
        sum(label != -100 for label in example["labels"])
        for example in dataset.examples
    )

    assert len(dataset) == expected_sentences
    assert labeled_words == expected_words

    print(
        f"[OK] {name}: "
        f"{len(dataset):,} sentences | "
        f"{labeled_words:,} labeled words"
    )


# --------------------------------------------
# 4. Configure dynamic padding
# --------------------------------------------

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
    return_tensors="pt"
)

sample_batch = data_collator([
    train_dataset[i]
    for i in range(8)
])

assert (
    sample_batch["input_ids"].shape
    == sample_batch["attention_mask"].shape
    == sample_batch["labels"].shape
)

assert sample_batch["input_ids"].shape[1] <= 256

print("\n=== GPU BATCH ===")
print("Input shape:", tuple(sample_batch["input_ids"].shape))
print("Label shape:", tuple(sample_batch["labels"].shape))


# --------------------------------------------
# 5. Verify mixed-precision GPU computation
# --------------------------------------------

gpu_batch = {
    key: value.to(device)
    for key, value in sample_batch.items()
}

bert_model.eval()

with torch.inference_mode():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):

        outputs = bert_model(**gpu_batch)

assert torch.isfinite(outputs.loss).item()

print("\n=== MIXED PRECISION VERIFICATION ===")
print("FP16 GPU computation: SUCCESS")
print("Output shape:", tuple(outputs.logits.shape))
print("Initial loss:", round(outputs.loss.item(), 4))
print("CUDA device:", torch.cuda.get_device_name(0))

del gpu_batch, outputs

print("\n=== CHECKPOINT STORAGE ===")
print("Google Drive:", PROJECT_DIR)
print("Persistent checkpoints: ENABLED")

print("\nCOLAB TRAINING INPUTS AND DRIVE READY")
print("Training started: NO")
print("Test datasets: NOT USED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 02
# COLAB CELL 5: PARSBERT GPU FINE-TUNING
# ============================================

import json
import math
import pandas as pd
import numpy as np
import torch

from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    Trainer,
    TrainingArguments
)
from transformers.trainer_utils import get_last_checkpoint

# --------------------------------------------
# 1. Training configuration
# --------------------------------------------

OUTPUT_DIR = CHECKPOINT_DIR / "parsbert_formal"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    gradient_accumulation_steps=2,

    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    lr_scheduler_type="linear",

    fp16=True,

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=100,

    max_grad_norm=1.0,

    seed=42,
    data_seed=42,

    dataloader_num_workers=0,

    report_to="none",

    save_safetensors=True
)

# --------------------------------------------
# 2. Token-level evaluation metrics
# --------------------------------------------

def compute_pos_metrics(eval_prediction):

    logits = eval_prediction.predictions
    labels = eval_prediction.label_ids

    predictions = np.argmax(
        logits,
        axis=-1
    )

    valid = labels != -100

    true_labels = labels[valid]
    predicted_labels = predictions[valid]

    accuracy = accuracy_score(
        true_labels,
        predicted_labels
    )

    macro_f1 = f1_score(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_TAGS)),
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_TAGS)),
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1)
    }


# --------------------------------------------
# 3. Initialize Hugging Face Trainer
# --------------------------------------------

trainer = Trainer(
    model=bert_model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=formal_eval_dataset,

    data_collator=data_collator,
    processing_class=tokenizer,

    compute_metrics=compute_pos_metrics
)

steps_per_epoch = math.ceil(
    math.ceil(len(train_dataset) / 8) / 2
)

print("=== PARSBERT TRAINING CONFIGURATION ===")
print("Model:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print("Training sentences:", len(train_dataset))
print("Formal validation sentences:", len(formal_eval_dataset))
print("POS categories:", NUM_TAGS)
print("Epochs:", training_args.num_train_epochs)
print("Effective batch size:", 16)
print("Optimizer steps per epoch:", steps_per_epoch)
print("Evaluation frequency:", 500, "steps")
print("Checkpoint directory:", OUTPUT_DIR)
print("FP16 mixed precision:", training_args.fp16)

# --------------------------------------------
# 4. Resume if a checkpoint exists
# --------------------------------------------

last_checkpoint = get_last_checkpoint(
    str(OUTPUT_DIR)
)

if last_checkpoint:
    print("\nRESUMING FROM:", last_checkpoint)
else:
    print("\nSTARTING NEW TRAINING RUN")

# --------------------------------------------
# 5. Start GPU training
# --------------------------------------------

print("\n=== STARTING PARSBERT FINE-TUNING ===")

train_result = trainer.train(
    resume_from_checkpoint=last_checkpoint
)

# --------------------------------------------
# 6. Save training history
# --------------------------------------------

history_path = (
    RESULTS_DIR /
    "parsbert_training_history.csv"
)

pd.DataFrame(
    trainer.state.log_history
).to_csv(
    history_path,
    index=False
)

# --------------------------------------------
# 7. Evaluate selected best model
# --------------------------------------------

print("\n=== BEST MODEL: FORMAL VALIDATION ===")

best_formal_results = trainer.evaluate(
    eval_dataset=formal_eval_dataset
)

print(
    "Accuracy:",
    round(best_formal_results["eval_accuracy"], 4)
)

print(
    "Macro F1:",
    round(best_formal_results["eval_macro_f1"], 4)
)

print(
    "Weighted F1:",
    round(best_formal_results["eval_weighted_f1"], 4)
)

print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint
)

# --------------------------------------------
# 8. Save experiment summary
# --------------------------------------------

summary = {
    "model": MODEL_NAME,
    "training_sentences": len(train_dataset),
    "validation_sentences": len(formal_eval_dataset),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_accuracy": best_formal_results[
        "eval_accuracy"
    ],
    "best_validation_macro_f1": best_formal_results[
        "eval_macro_f1"
    ],
    "epochs_requested": 3,
    "seed": 42,
    "informal_used_for_selection": False,
    "test_sets_used": False
}

summary_path = (
    RESULTS_DIR /
    "parsbert_experiment_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n=== TRAINING COMPLETE ===")
print("Training history:", history_path)
print("Experiment summary:", summary_path)
print("Best model saved on Google Drive.")
print("Informal validation: NOT USED FOR SELECTION")
print("Test datasets: NOT USED")

print("\nPARSBERT FINE-TUNING COMPLETE")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 02
# COLAB CELL 6: CROSS-REGISTER EVALUATION
# ============================================

import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# --------------------------------------------
# 1. Confirm best checkpoint is selected
# --------------------------------------------

print("=== SELECTED PARSBERT MODEL ===")
print("Best checkpoint:", trainer.state.best_model_checkpoint)

assert trainer.state.best_model_checkpoint is not None

# Trainer restored the best model automatically
# because load_best_model_at_end=True.
trainer.model.eval()

# --------------------------------------------
# 2. Evaluate informal Persian
# --------------------------------------------

print("\n=== INFORMAL PERSIAN EVALUATION ===")

prediction_output = trainer.predict(
    informal_eval_dataset,
    metric_key_prefix="informal"
)

labels = prediction_output.label_ids
predictions = np.argmax(
    prediction_output.predictions,
    axis=-1
)

valid = labels != -100

y_true = labels[valid]
y_pred = predictions[valid]

informal_accuracy = accuracy_score(
    y_true,
    y_pred
)

informal_macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=list(range(NUM_TAGS)),
    average="macro",
    zero_division=0
)

print("Evaluated words:", len(y_true))
print(f"Token accuracy: {informal_accuracy:.4f}")
print(f"Macro F1: {informal_macro_f1:.4f}")

# --------------------------------------------
# 3. Compare against BiLSTM
# --------------------------------------------

formal_accuracy = best_formal_results["eval_accuracy"]
formal_macro_f1 = best_formal_results["eval_macro_f1"]

comparison = pd.DataFrame([
    {
        "Model": "BiLSTM",
        "Formal Accuracy": 0.9545,
        "Informal Accuracy": 0.7895,
        "Formal Macro F1": 0.8809,
        "Informal Macro F1": 0.6555
    },
    {
        "Model": "ParsBERT",
        "Formal Accuracy": formal_accuracy,
        "Informal Accuracy": informal_accuracy,
        "Formal Macro F1": formal_macro_f1,
        "Informal Macro F1": informal_macro_f1
    }
])

comparison["Accuracy Gap (pp)"] = (
    comparison["Formal Accuracy"]
    - comparison["Informal Accuracy"]
) * 100

comparison["Macro F1 Gap"] = (
    comparison["Formal Macro F1"]
    - comparison["Informal Macro F1"]
)

print("\n=== BILSTM VS PARSBERT ===")
print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# --------------------------------------------
# 4. Per-category informal POS performance
# --------------------------------------------

tag_names = [
    id2pos[i]
    for i in range(NUM_TAGS)
]

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_TAGS)),
    target_names=tag_names,
    output_dict=True,
    zero_division=0
)

pos_results = pd.DataFrame([
    {
        "POS": tag,
        "Precision": report[tag]["precision"],
        "Recall": report[tag]["recall"],
        "F1": report[tag]["f1-score"],
        "Support": int(report[tag]["support"])
    }
    for tag in tag_names
])

print("\n=== PARSBERT INFORMAL POS PERFORMANCE ===")
print(
    pos_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# --------------------------------------------
# 5. Save findings to Google Drive
# --------------------------------------------

comparison_path = (
    RESULTS_DIR / "bilstm_vs_parsbert.csv"
)

pos_path = (
    RESULTS_DIR / "parsbert_informal_pos_results.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)

pos_results.to_csv(
    pos_path,
    index=False
)

print("\n=== RESULTS SAVED ===")
print("Comparison:", comparison_path)
print("POS performance:", pos_path)

print("\nPARSBERT CROSS-REGISTER EVALUATION COMPLETE")
print("Model weights: UNCHANGED")
print("Test datasets: NOT USED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 02
# COLAB CELL 7: ANNOTATION CONSISTENCY AUDIT
# ============================================

from collections import Counter, defaultdict
from pathlib import Path
from urllib.request import urlretrieve
import pandas as pd

# --------------------------------------------
# 1. Download original annotation files
# --------------------------------------------

RAW_DIR = Path("/content/farsi-syntax-lab/data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

sources = {
    "Formal train": (
        "https://raw.githubusercontent.com/"
        "UniversalDependencies/UD_Persian-PerDT/"
        "master/fa_perdt-ud-train.conllu"
    ),
    "Informal dev": (
        "https://raw.githubusercontent.com/"
        "royakabiri/iPerUDT/"
        "main/iPerUDT_dev.conllu"
    )
}

files = {}

for name, url in sources.items():

    path = RAW_DIR / (
        "formal_train.conllu"
        if name == "Formal train"
        else "informal_dev.conllu"
    )

    if not path.exists():
        print("Downloading:", name)
        urlretrieve(url, path)

    assert path.stat().st_size > 10000

    files[name] = path

    print(f"[OK] {name}: {path.stat().st_size:,} bytes")


# --------------------------------------------
# 2. Extract word-level POS annotations
# --------------------------------------------

def read_word_annotations(path):

    word_tags = defaultdict(Counter)
    tag_counts = Counter()
    total_words = 0

    with path.open("r", encoding="utf-8-sig") as f:

        for line in f:

            if line.startswith("#") or not line.strip():
                continue

            columns = line.rstrip("\n").split("\t")

            if len(columns) != 10:
                continue

            if not columns[0].isdigit():
                continue

            word = columns[1]
            tag = columns[3]

            word_tags[word][tag] += 1
            tag_counts[tag] += 1
            total_words += 1

    return word_tags, tag_counts, total_words


formal_tags, formal_counts, formal_total = (
    read_word_annotations(files["Formal train"])
)

informal_tags, informal_counts, informal_total = (
    read_word_annotations(files["Informal dev"])
)

assert formal_total == 452496
assert informal_total == 10049

print("\n=== CORPUS ANNOTATION COUNTS ===")

for tag in ["PART", "X", "AUX", "ADV", "VERB"]:

    print(
        f"{tag:5s} | "
        f"Formal: {formal_counts[tag]:6,d} | "
        f"Informal: {informal_counts[tag]:5,d}"
    )


# --------------------------------------------
# 3. Compare words labeled PART in informal
# --------------------------------------------

informal_particles = Counter({
    word: labels["PART"]
    for word, labels in informal_tags.items()
    if labels["PART"] > 0
})

records = []

for word, part_count in informal_particles.most_common(20):

    formal_labels = formal_tags.get(word, Counter())
    informal_labels = informal_tags[word]

    records.append({
        "Word": word,
        "Informal PART Count": part_count,
        "Formal Total": sum(formal_labels.values()),
        "Formal POS Distribution": str(
            formal_labels.most_common()
        ),
        "Informal POS Distribution": str(
            informal_labels.most_common()
        )
    })

comparison_df = pd.DataFrame(records)

print("\n=== SAME WORD, DIFFERENT ANNOTATIONS ===")

print(
    comparison_df.to_string(
        index=False
    )
)


# --------------------------------------------
# 4. Save research evidence
# --------------------------------------------

audit_path = (
    RESULTS_DIR / "persian_annotation_consistency.csv"
)

comparison_df.to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n=== AUDIT COMPLETE ===")
print("Saved:", audit_path)
print("Training: NOT MODIFIED")
print("Checkpoints: UNCHANGED")
print("Test datasets: NOT USED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 03
# COLAB CELL 8: HELD-OUT TEST DATA AUDIT
# ============================================

from pathlib import Path
from urllib.request import urlretrieve
from collections import Counter
import hashlib
import numpy as np
import pandas as pd

# --------------------------------------------
# 1. Official test dataset sources
# --------------------------------------------

TEST_DIR = PROJECT_DIR / "data" / "raw"
TEST_DIR.mkdir(parents=True, exist_ok=True)

test_sources = {
    "Formal test": (
        "https://raw.githubusercontent.com/"
        "UniversalDependencies/UD_Persian-PerDT/"
        "master/fa_perdt-ud-test.conllu"
    ),
    "Informal test": (
        "https://raw.githubusercontent.com/"
        "royakabiri/iPerUDT/"
        "main/iPerUDT_test.conllu"
    )
}

test_paths = {}

for name, url in test_sources.items():

    filename = name.lower().replace(" ", "_") + ".conllu"
    path = TEST_DIR / filename

    if not path.exists():
        print("Downloading:", name)
        urlretrieve(url, path)

    assert path.stat().st_size > 10000

    test_paths[name] = path

    print(f"[OK] {name}: {path.stat().st_size:,} bytes")


# --------------------------------------------
# 2. Read CoNLL-U sentences without alteration
# --------------------------------------------

def inspect_conllu(path):

    sentences = []
    current = []
    malformed = 0

    with path.open("r", encoding="utf-8-sig") as file:

        for raw in file:

            line = raw.strip()

            if not line:

                if current:
                    sentences.append(current)
                    current = []

                continue

            if line.startswith("#"):
                continue

            columns = line.split("\t")

            if len(columns) != 10:
                malformed += 1
                continue

            # Exclude multiword ranges / empty nodes
            if not columns[0].isdigit():
                continue

            current.append(
                (columns[1], columns[3])
            )

    if current:
        sentences.append(current)

    return sentences, malformed


# --------------------------------------------
# 3. Complete tokenization audit
# --------------------------------------------

records = []

print("\n=== OFFICIAL TEST SPLIT AUDIT ===")

for name, path in test_paths.items():

    sentences, malformed = inspect_conllu(path)

    pos_counts = Counter(
        tag
        for sentence in sentences
        for word, tag in sentence
    )

    unseen_tags = set(pos_counts) - set(pos2id)

    assert malformed == 0
    assert not unseen_tags, (
        f"Unseen POS labels: {unseen_tags}"
    )

    lengths = []
    alignment_failures = 0

    for start in range(0, len(sentences), 256):

        batch = sentences[start:start + 256]

        word_lists = [
            [word for word, tag in sentence]
            for sentence in batch
        ]

        encoded = tokenizer(
            word_lists,
            is_split_into_words=True,
            truncation=False,
            padding=False,
            return_length=True
        )

        lengths.extend(encoded["length"])

        for i, words in enumerate(word_lists):

            covered = {
                word_id
                for word_id in encoded.word_ids(i)
                if word_id is not None
            }

            if len(covered) != len(words):
                alignment_failures += 1

    lengths = np.asarray(lengths)

    checksum = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    records.append({
        "Dataset": name,
        "Sentences": len(sentences),
        "Labeled words": sum(pos_counts.values()),
        "POS categories": len(pos_counts),
        "Max subwords": int(lengths.max()),
        "Over 256": int((lengths > 256).sum()),
        "Over 512": int((lengths > 512).sum()),
        "Alignment failures": alignment_failures,
        "SHA256": checksum
    })


# --------------------------------------------
# 4. Display integrity report
# --------------------------------------------

audit_df = pd.DataFrame(records)

print("\n=== OFFICIAL TEST DATA REPORT ===")

print(
    audit_df.drop(columns=["SHA256"]).to_string(
        index=False
    )
)

assert (
    audit_df["Alignment failures"].sum() == 0
), "Alignment issue detected"

report_path = (
    RESULTS_DIR / "official_test_data_audit.csv"
)

audit_df.to_csv(report_path, index=False)

print("\n=== TEST DATA STATUS ===")
print("Official annotations: UNCHANGED")
print("Test labels inspected for integrity only")
print("Model predictions: NOT GENERATED")
print("Model checkpoints: UNCHANGED")
print("Saved:", report_path)

print("\nOFFICIAL TEST DATA AUDIT COMPLETE")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 03
# COLAB CELL 9: OFFICIAL PARSBERT TEST
# ============================================

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

MAX_LENGTH = 256
IGNORE_INDEX = -100

# --------------------------------------------
# 1. Confirm selected model
# --------------------------------------------

assert trainer.state.best_model_checkpoint is not None

trainer.model.eval()

print("=== OFFICIAL TEST EVALUATION ===")
print("Model:", MODEL_NAME)
print(
    "Selected checkpoint:",
    trainer.state.best_model_checkpoint
)
print("Device:", next(trainer.model.parameters()).device)


# --------------------------------------------
# 2. Prepare test datasets
# --------------------------------------------

class OfficialTestDataset(Dataset):

    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def prepare_official_test(path, expected_words):

    sentences, malformed = inspect_conllu(path)

    assert malformed == 0

    examples = []
    total_labels = 0

    for start in range(0, len(sentences), 128):

        batch = sentences[start:start + 128]

        words_batch = [
            [word for word, tag in sentence]
            for sentence in batch
        ]

        tags_batch = [
            [tag for word, tag in sentence]
            for sentence in batch
        ]

        encoded = tokenizer(
            words_batch,
            is_split_into_words=True,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False
        )

        for i, tags in enumerate(tags_batch):

            word_ids = encoded.word_ids(batch_index=i)

            labels = []
            previous_word_id = None

            for word_id in word_ids:

                if word_id is None:
                    labels.append(IGNORE_INDEX)

                elif word_id != previous_word_id:
                    labels.append(pos2id[tags[word_id]])

                else:
                    labels.append(IGNORE_INDEX)

                previous_word_id = word_id

            supervised = sum(
                label != IGNORE_INDEX
                for label in labels
            )

            # Prevent any silent annotation loss
            assert supervised == len(tags)

            total_labels += supervised

            examples.append({
                "input_ids": encoded["input_ids"][i],
                "attention_mask": encoded[
                    "attention_mask"
                ][i],
                "labels": labels
            })

    assert total_labels == expected_words

    return OfficialTestDataset(examples)


formal_test_dataset = prepare_official_test(
    test_paths["Formal test"],
    expected_words=24133
)

informal_test_dataset = prepare_official_test(
    test_paths["Informal test"],
    expected_words=10014
)

print("\n=== TEST DATA PREPARED ===")
print("Formal sentences:", len(formal_test_dataset))
print("Informal sentences:", len(informal_test_dataset))
print("All annotations preserved: YES")


# --------------------------------------------
# 3. Run official test predictions
# --------------------------------------------

test_results = []
per_pos_results = []

for register, dataset, expected_words in [
    ("Formal", formal_test_dataset, 24133),
    ("Informal", informal_test_dataset, 10014)
]:

    print(f"\n=== EVALUATING {register.upper()} TEST ===")

    output = trainer.predict(
        dataset,
        metric_key_prefix=register.lower() + "_test"
    )

    predictions = np.argmax(
        output.predictions,
        axis=-1
    )

    labels = output.label_ids
    valid = labels != IGNORE_INDEX

    y_true = labels[valid]
    y_pred = predictions[valid]

    assert len(y_true) == expected_words

    accuracy = accuracy_score(y_true, y_pred)

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(range(NUM_TAGS)),
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(range(NUM_TAGS)),
        average="weighted",
        zero_division=0
    )

    test_results.append({
        "Model": "ParsBERT",
        "Register": register,
        "Tokens": len(y_true),
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1
    })

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_TAGS)),
        target_names=[
            id2pos[i] for i in range(NUM_TAGS)
        ],
        output_dict=True,
        zero_division=0
    )

    for i in range(NUM_TAGS):

        tag = id2pos[i]
        tag_report = report[tag]

        per_pos_results.append({
            "Register": register,
            "POS": tag,
            "Precision": tag_report["precision"],
            "Recall": tag_report["recall"],
            "F1": tag_report["f1-score"],
            "Support": int(tag_report["support"])
        })

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")


# --------------------------------------------
# 4. Official result summary
# --------------------------------------------

test_df = pd.DataFrame(test_results)

print("\n=== OFFICIAL PARSBERT TEST RESULTS ===")

print(
    test_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

formal_row = test_df[
    test_df["Register"] == "Formal"
].iloc[0]

informal_row = test_df[
    test_df["Register"] == "Informal"
].iloc[0]

accuracy_gap = (
    formal_row["Accuracy"]
    - informal_row["Accuracy"]
) * 100

macro_gap = (
    formal_row["Macro F1"]
    - informal_row["Macro F1"]
)

print("\n=== OFFICIAL CROSS-REGISTER GAP ===")
print(f"Accuracy gap: {accuracy_gap:.2f} percentage points")
print(f"Macro F1 gap: {macro_gap:.4f}")


# --------------------------------------------
# 5. Save final results to Google Drive
# --------------------------------------------

test_output = (
    RESULTS_DIR / "parsbert_official_test_results.csv"
)

pos_output = (
    RESULTS_DIR / "parsbert_official_test_per_pos.csv"
)

test_df.to_csv(test_output, index=False)

pd.DataFrame(per_pos_results).to_csv(
    pos_output,
    index=False
)

print("\n=== RESULTS SAVED ===")
print("Test summary:", test_output)
print("Per-category metrics:", pos_output)

print("\nOFFICIAL PARSBERT TEST EVALUATION COMPLETE")
print("Model weights: UNCHANGED")
print("Further model selection on test results: NOT PERMITTED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 03
# COLAB CELL 10: RESTORE TRAINED BILSTM
# ============================================

from google.colab import files
from pathlib import Path
from torch import nn
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)
import torch

# --------------------------------------------
# 1. Upload existing trained checkpoint
# --------------------------------------------

print("Select bilstm_formal_best.pt from Windows")

uploaded = files.upload()

filename = "bilstm_formal_best.pt"

assert filename in uploaded, (
    "Please upload bilstm_formal_best.pt"
)

# Save permanently to Google Drive
BILSTM_PATH = (
    CHECKPOINT_DIR / "bilstm_formal_best.pt"
)

BILSTM_PATH.write_bytes(uploaded[filename])

# --------------------------------------------
# 2. Read original model configuration
# --------------------------------------------

checkpoint = torch.load(
    BILSTM_PATH,
    map_location="cpu",
    weights_only=True
)

bilstm_word2id = checkpoint["word2id"]
bilstm_pos2id = checkpoint["pos2id"]

assert bilstm_pos2id == pos2id, (
    "BiLSTM and ParsBERT POS mappings differ!"
)

assert checkpoint["best_epoch"] == 9

print("\n=== BILSTM CHECKPOINT ===")
print("Best epoch:", checkpoint["best_epoch"])
print("Vocabulary:", len(bilstm_word2id))
print("POS categories:", len(bilstm_pos2id))
print(
    "Formal validation Macro F1:",
    checkpoint["validation_macro_f1"]
)

# --------------------------------------------
# 3. Reconstruct the exact trained architecture
# --------------------------------------------

class PersianBiLSTMTagger(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_tags,
        pad_id,
        dropout
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_id
        )

        self.dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )

        self.classifier = nn.Linear(
            hidden_dim * 2,
            num_tags
        )

    def forward(self, word_ids, lengths):

        embedded = self.dropout(
            self.embedding(word_ids)
        )

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=word_ids.size(1)
        )

        return self.classifier(
            self.dropout(output)
        )


bilstm_model = PersianBiLSTMTagger(
    vocab_size=len(bilstm_word2id),
    embedding_dim=checkpoint["embedding_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    num_tags=len(bilstm_pos2id),
    pad_id=bilstm_word2id["<PAD>"],
    dropout=checkpoint["dropout"]
).to(device)

# Restore trained weights
bilstm_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

bilstm_model.eval()

# --------------------------------------------
# 4. Verify inference without using test data
# --------------------------------------------

sample_words = [
    "من",
    "کتاب",
    "می‌خوانم",
    "."
]

sample_ids = torch.tensor(
    [[
        bilstm_word2id.get(
            word,
            bilstm_word2id["<UNK>"]
        )
        for word in sample_words
    ]],
    dtype=torch.long,
    device=device
)

sample_lengths = torch.tensor([len(sample_words)])

with torch.inference_mode():

    logits = bilstm_model(
        sample_ids,
        sample_lengths
    )

assert logits.shape == (1, 4, 16)
assert torch.isfinite(logits).all().item()

# --------------------------------------------
# 5. Final verification
# --------------------------------------------

parameters = sum(
    p.numel()
    for p in bilstm_model.parameters()
)

assert parameters == 2396560

print("\n=== BILSTM RESTORATION VERIFICATION ===")
print("Parameters:", f"{parameters:,}")
print("Device:", next(bilstm_model.parameters()).device)
print("Forward pass:", tuple(logits.shape))
print("Checkpoint restored: YES")
print("POS label mappings match: YES")
print("Model retrained: NO")
print("Test datasets evaluated: NO")

print("\nBILSTM RESTORATION COMPLETE")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 03
# COLAB CELL 11: FINAL MODEL COMPARISON
# ============================================

import torch
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# --------------------------------------------
# 1. Prepare BiLSTM test dataset
# --------------------------------------------

class BiLSTMTestDataset(Dataset):

    def __init__(self, sentences):
        self.sentences = sentences

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, index):

        sentence = self.sentences[index]

        word_ids = [
            bilstm_word2id.get(
                word,
                bilstm_word2id["<UNK>"]
            )
            for word, tag in sentence
        ]

        label_ids = [
            bilstm_pos2id[tag]
            for word, tag in sentence
        ]

        return (
            torch.tensor(word_ids, dtype=torch.long),
            torch.tensor(label_ids, dtype=torch.long)
        )


def bilstm_collate(batch):

    words, labels = zip(*batch)

    lengths = torch.tensor(
        [len(sentence) for sentence in words],
        dtype=torch.long
    )

    padded_words = pad_sequence(
        words,
        batch_first=True,
        padding_value=bilstm_word2id["<PAD>"]
    )

    padded_labels = pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100
    )

    return padded_words, padded_labels, lengths


# --------------------------------------------
# 2. Frozen BiLSTM evaluation
# --------------------------------------------

@torch.inference_mode()
def evaluate_bilstm_test(sentences):

    dataset = BiLSTMTestDataset(sentences)

    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=False,
        collate_fn=bilstm_collate,
        num_workers=0
    )

    bilstm_model.eval()

    true_labels = []
    predicted_labels = []
    oov_flags = []

    for words, labels, lengths in loader:

        logits = bilstm_model(
            words.to(device),
            lengths
        )

        predictions = logits.argmax(
            dim=-1
        ).cpu()

        valid = labels != -100

        true_labels.extend(
            labels[valid].tolist()
        )

        predicted_labels.extend(
            predictions[valid].tolist()
        )

        oov_flags.extend(
            (
                words[valid]
                == bilstm_word2id["<UNK>"]
            ).tolist()
        )

    y_true = np.asarray(true_labels)
    y_pred = np.asarray(predicted_labels)
    is_oov = np.asarray(oov_flags, dtype=bool)

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(
            y_true,
            y_pred,
            labels=list(range(16)),
            average="macro",
            zero_division=0
        ),
        "Weighted F1": f1_score(
            y_true,
            y_pred,
            labels=list(range(16)),
            average="weighted",
            zero_division=0
        ),
        "OOV Rate": float(is_oov.mean())
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(16)),
        target_names=[
            id2pos[i] for i in range(16)
        ],
        output_dict=True,
        zero_division=0
    )

    return metrics, report, len(y_true)


# --------------------------------------------
# 3. Evaluate both official test sets
# --------------------------------------------

print("=== OFFICIAL BILSTM TEST EVALUATION ===")

bilstm_results = []
bilstm_pos_results = []

test_configs = [
    ("Formal", test_paths["Formal test"], 24133),
    ("Informal", test_paths["Informal test"], 10014)
]

for register, path, expected_words in test_configs:

    sentences, malformed = inspect_conllu(path)

    assert malformed == 0

    metrics, report, count = evaluate_bilstm_test(
        sentences
    )

    assert count == expected_words

    bilstm_results.append({
        "Model": "BiLSTM",
        "Register": register,
        "Tokens": count,
        **metrics
    })

    for tag in [id2pos[i] for i in range(16)]:

        bilstm_pos_results.append({
            "Register": register,
            "POS": tag,
            "Precision": report[tag]["precision"],
            "Recall": report[tag]["recall"],
            "F1": report[tag]["f1-score"],
            "Support": int(report[tag]["support"])
        })

    print(f"\n=== {register.upper()} TEST ===")
    print("Sentences:", len(sentences))
    print("Words:", count)
    print(f"Accuracy: {metrics['Accuracy']:.4f}")
    print(f"Macro F1: {metrics['Macro F1']:.4f}")
    print(f"Weighted F1: {metrics['Weighted F1']:.4f}")
    print(f"OOV rate: {metrics['OOV Rate']:.2%}")


# --------------------------------------------
# 4. Load previously saved ParsBERT results
# --------------------------------------------

parsbert_path = (
    RESULTS_DIR / "parsbert_official_test_results.csv"
)

assert parsbert_path.exists(), (
    "Saved ParsBERT test results not found."
)

parsbert_results = pd.read_csv(parsbert_path)

bilstm_df = pd.DataFrame(bilstm_results)

# Ensure both models use the same test token counts
for register, _, expected_words in test_configs:

    bert_count = parsbert_results.loc[
        parsbert_results["Register"] == register,
        "Tokens"
    ].iloc[0]

    assert int(bert_count) == expected_words


# --------------------------------------------
# 5. Final official comparison
# --------------------------------------------

comparison = pd.concat(
    [
        bilstm_df.drop(columns=["OOV Rate"]),
        parsbert_results
    ],
    ignore_index=True
)

print("\n=== FINAL OFFICIAL TEST COMPARISON ===")

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# --------------------------------------------
# 6. Calculate improvement and register gaps
# --------------------------------------------

print("\n=== MODEL IMPROVEMENT ===")

for register in ["Formal", "Informal"]:

    rows = comparison[
        comparison["Register"] == register
    ].set_index("Model")

    accuracy_change = (
        rows.loc["ParsBERT", "Accuracy"]
        - rows.loc["BiLSTM", "Accuracy"]
    ) * 100

    f1_change = (
        rows.loc["ParsBERT", "Macro F1"]
        - rows.loc["BiLSTM", "Macro F1"]
    )

    print(
        f"{register}: "
        f"Accuracy improvement = "
        f"{accuracy_change:+.2f} percentage points | "
        f"Macro F1 improvement = {f1_change:+.4f}"
    )


print("\n=== FORMAL-TO-INFORMAL PERFORMANCE GAP ===")

for model_name in ["BiLSTM", "ParsBERT"]:

    rows = comparison[
        comparison["Model"] == model_name
    ].set_index("Register")

    accuracy_gap = (
        rows.loc["Formal", "Accuracy"]
        - rows.loc["Informal", "Accuracy"]
    ) * 100

    f1_gap = (
        rows.loc["Formal", "Macro F1"]
        - rows.loc["Informal", "Macro F1"]
    )

    print(
        f"{model_name}: "
        f"Accuracy gap = {accuracy_gap:.2f} pp | "
        f"Macro F1 gap = {f1_gap:.4f}"
    )


# --------------------------------------------
# 7. Save final research results
# --------------------------------------------

bilstm_df.to_csv(
    RESULTS_DIR / "bilstm_official_test_results.csv",
    index=False
)

pd.DataFrame(bilstm_pos_results).to_csv(
    RESULTS_DIR / "bilstm_official_test_per_pos.csv",
    index=False
)

comparison.to_csv(
    RESULTS_DIR / "final_model_comparison.csv",
    index=False
)

print("\n=== FINAL RESULTS SAVED ===")
print(
    "Comparison:",
    RESULTS_DIR / "final_model_comparison.csv"
)

print("\nEXPERIMENT 03 - FINAL COMPARISON COMPLETE")
print("BiLSTM retrained: NO")
print("ParsBERT retrained: NO")
print("Checkpoints modified: NO")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 04
# COLAB CELL 12: EXPORT DEPLOYABLE MODEL
# ============================================

from pathlib import Path
import json
import torch

from transformers import AutoConfig, AutoTokenizer

# --------------------------------------------
# 1. Deployment directory
# --------------------------------------------

DEPLOY_DIR = (
    PROJECT_DIR / "deployment" / "parsbert-pos"
)

DEPLOY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------
# 2. Verify selected checkpoint
# --------------------------------------------

best_checkpoint = trainer.state.best_model_checkpoint

assert best_checkpoint is not None
assert Path(best_checkpoint).name == "checkpoint-4000"

assert trainer.model.config.num_labels == 16

print("=== DEPLOYMENT EXPORT ===")
print("Source:", best_checkpoint)
print("Destination:", DEPLOY_DIR)

# --------------------------------------------
# 3. Save the selected trained model
# --------------------------------------------

trainer.save_model(str(DEPLOY_DIR))

# Explicitly save the Persian tokenizer
tokenizer.save_pretrained(str(DEPLOY_DIR))

print("\nModel and tokenizer saved.")

# --------------------------------------------
# 4. Save research metadata
# --------------------------------------------

model_metadata = {
    "project": "FarsiSyntax Lab",
    "base_model": MODEL_NAME,
    "architecture": "ParsBERT POS Tagger",
    "source_checkpoint": str(best_checkpoint),
    "num_labels": 16,
    "id2label": {
        str(i): id2pos[i]
        for i in range(16)
    },
    "training_corpus": "UD Persian-PerDT",
    "cross_register_corpus": "iPerUDT",
    "formal_test_accuracy": 0.9727,
    "formal_test_macro_f1": 0.8982,
    "informal_test_accuracy": 0.8736,
    "informal_test_macro_f1": 0.7334,
    "seed": 42,
    "limitations": [
        "Cross-treebank POS annotation differences",
        "Reduced accuracy on conversational Persian",
        "PART and X category difficulties",
        "POS tagging only; not dependency parsing"
    ]
}

with open(
    DEPLOY_DIR / "research_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        model_metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

# --------------------------------------------
# 5. Verify exported model configuration
# --------------------------------------------

saved_config = AutoConfig.from_pretrained(
    str(DEPLOY_DIR)
)

saved_tokenizer = AutoTokenizer.from_pretrained(
    str(DEPLOY_DIR),
    use_fast=True
)

assert saved_config.num_labels == 16
assert saved_tokenizer.vocab_size == 42000

for i in range(16):
    assert saved_config.id2label[i] == id2pos[i]

weight_files = (
    list(DEPLOY_DIR.glob("*.safetensors"))
    + list(DEPLOY_DIR.glob("pytorch_model.bin"))
)

assert weight_files, "Model weights missing"

for weight_file in weight_files:
    assert weight_file.stat().st_size > 100_000_000

# --------------------------------------------
# 6. Display exported files
# --------------------------------------------

print("\n=== EXPORTED MODEL FILES ===")

total_bytes = 0

for path in sorted(DEPLOY_DIR.iterdir()):

    if path.is_file():

        size = path.stat().st_size
        total_bytes += size

        print(
            f"{path.name:35s} "
            f"{size / 1024**2:9.2f} MB"
        )

print("\n=== DEPLOYMENT VERIFICATION ===")
print("Classification labels:", saved_config.num_labels)
print("Tokenizer vocabulary:", saved_tokenizer.vocab_size)
print("Trained weights:", "PRESENT")
print("Source checkpoint:", Path(best_checkpoint).name)
print("Export size:", round(total_bytes / 1024**2, 2), "MB")
print("Location:", DEPLOY_DIR)

print("\nPARSBERT DEPLOYMENT MODEL READY")
print("Model retrained: NO")
print("Original checkpoint: UNCHANGED")
